In [1]:
# Install
# %pip install -e ..
# This kiiinda works?
%load_ext autoreload

In [6]:
# Imports and Setup
%autoreload 2
import logging
import ezregex as ez
import re
from ezregex.invert import Inverter
import re._compiler
import re._parser as sre
import re._parser
from pprint import pformat
from rich.pretty import pprint
from collections.abc import Mapping

logging.basicConfig(level=logging.DEBUG)
original_handle = Inverter._handle


In [19]:
from attr import dataclass


@dataclass
class G:
    # Just combine these
    grouping = 'Grouping'
    advanced_grouping = 'Grouping'
    replacements = 'Replacements'
    advanced_replacements = 'Replacements'
    assertions = 'Assertions'
    anchors = 'Anchors'
    base = 'Base'
    premade = 'Premade'
    lang_specific = 'Dialect Specific'
    amounts = 'Amounts'
    common = 'Common'

group_colors = {
    G.grouping: 20,
    G.advanced_grouping: 20,
    G.replacements: 0,
    G.advanced_replacements: 0,
    G.assertions: 230,
    G.anchors: 160,
    G.base: 330,
    G.premade: 260,
    G.lang_specific: 330,
    G.amounts: 120,
    G.common: 210
}

groups = {
    'any_between': G.base,
    'any_char_except': G.base,
    'any_of': G.base,
    'anything': G.base,
    'at_least_none': G.amounts,
    'at_least_one': G.amounts,
    'carriage_return': G.base,
    'chunk': G.common,
    'comma': G.base,
    'controller': G.base,
    'digit': G.common,
    'each': G.assertions,
    'earlier_group': G.advanced_grouping,
    'either': G.common,
    'email': G.premade,
    'form_feed': G.base,
    'full_float': G.premade,
    'group': G.grouping,
    'hex_digit': G.base,
    'if_enclosed_with': G.assertions,
    'if_not_preceded_by': G.assertions,
    'if_not_proceded_by': G.assertions,
    'if_preceded_by': G.assertions,
    'if_proceded_by': G.assertions,
    'int_or_float': G.premade,
    'is_exactly': G.anchors,
    'letter': G.common,
    'letter_num': G.base,
    'line_ends_with': G.anchors,
    'line_starts_with': G.anchors,
    'literal': G.common,
    'literally_anything': G.premade,
    'lowercase': G.base,
    'match_at_least': G.amounts,
    'match_at_most': G.amounts,
    'match_max': G.amounts,
    'match_more_than': G.amounts,
    'match_num': G.amounts,
    'match_range': G.amounts,
    'new_line': G.base,
    'not_digit': G.base,
    'not_whitespace': G.base,
    'not_word': G.base,
    'not_word_boundary': G.anchors,
    'number': G.common,
    'oct_digit': G.base,
    'optional': G.base,
    'or': G.base,
    'ow': G.common,
    'passive_group': G.grouping,
    'period': G.base,
    'plain_float': G.premade,
    'printable': G.base,
    'printable_and_space': G.base,
    'punctuation': G.base,
    'quote': G.premade,
    'raw': G.common,
    'replace_entire': G.replacements,
    'rgroup': G.replacements,
    'rliteral': G.replacements,
    'signed_integer': G.premade,
    'signed_number': G.premade,
    'space': G.base,
    'space_or_tab': G.base,
    'string_ends_with': G.anchors,
    'string_starts_with': G.anchors,
    'tab': G.base,
    'underscore': G.base,
    'unicode': G.base,
    'unpadded_number': G.premade,
    'unsigned_integer': G.premade,
    'unsigned_number': G.premade,
    'uppercase': G.base,
    'version': G.base,
    'version_numbered': G.base,
    'vertical_tab': G.base,
    'white_char': G.base,
    'whitechunk': G.common,
    'word': G.common,
    'word_boundary': G.anchors,
    'word_char': G.base,
    'if_exists': G.advanced_grouping,
    'any_except': G.assertions,
    'word_starts_with': G.anchors,
    'word_ends_with': G.anchors,
    'replace': G.replacements,
    'entire_string': G.advanced_replacements,
    'string_before_match': G.advanced_replacements,
    'string_after_match': G.advanced_replacements
}

In [22]:
# Deleteme - for testing the ezregex-blockly generator
import inspect

from ezregex import generate

def generate_block_definitions(cls):
    elements = cls.parts(include_functions=False, include_psuedonyms=False)
    block_definitions = []
    for i in elements:
        obj = getattr(cls, i)
        funcs = len(obj._func_list)

        message0 = i.replace('_', ' ').title()
        args0 = []

        args = inspect.signature(obj._func_list[0]).parameters


        arg_count = 0
        def add_arg(name):
            nonlocal arg_count, message0
            message0 += f'{not arg_count and "\n"} {name}: %{arg_count}'
            arg_count += 1

        has_greedy_or_possessive = False
        for name, param in args.items():
            if name in ('greedy', 'possessive'):
                has_greedy_or_possessive = True

            if name in ('args', 'kwargs', 'cur'):
                pass
            # We can ignore chars and split parameters
            elif param.annotation == bool | None and name in ('chars', 'split'):
                pass
            elif name.endswith('pattern'):
                add_arg('')
                args0.append({
                    'type': 'input_statement',
                    'check': 'Regular',
                    'name': name.upper(),
                })
            elif param.annotation == bool:
                add_arg(name)
                args0.append({
                    'type': 'field_checkbox',
                    'name': name.upper(),
                    "checked": param.default,
                })
            elif (param.annotation == str
                # These 2 are lambdas, and I don't think you can add parameter type annotations to lambdas
                # (in a way you can pick them up with inpsect, anyway), so this works, whatever
                or (name in ('regex', 'name') and i in ('raw', 'unicode'))
                or (param.annotation == str | int and name == 'num_or_name')
            ):
                add_arg(name)
            elif param.annotation == int:
                add_arg(name)
            elif param.annotation == tuple[str]:
                add_arg(name)
            elif name == 'patterns':
                add_arg(name)
            else:
                print(i, ':', name, param)
                raise NotImplementedError


        to_add = {
            "type": i,
            'message0': message0,
            'previousStatement': 'Regular',
            'nextStatement': 'Regular',
            'colour': group_colors[groups[i]],
            'tooltip': obj.docstring,
            'helpUrl': None,
            "extensions": ["one_var_at_a_time"],
        }

        # Custom modifications
        if name in ('version', 'version_numbered'):
            to_add['helpUrl'] = 'https://semver.org/'

        if has_greedy_or_possessive:
            # By convention, this has the official docs for this dialect
            to_add['helpUrl'] = cls.__doc__

        if to_add['helpUrl'] is None:
            to_add.pop('helpUrl')

        if i not in ('group',):
            to_add.pop('extensions')

        if i in ('string_starts_with', 'if_preceded_by', 'if_not_preceded_by', 'is_exactly'):
            to_add['previousStatement'] = 'Beginning'

        if i in ('is_exactly', 'if_proceded_by', 'if_not_proceded_by', 'string_end'):
            to_add.pop('nextStatement')

        block_definitions.append(to_add)

    return block_definitions

import json
print(json.dumps(generate_block_definitions(ez.PythonEZRegex), indent=4))


[
    {
        "type": "any_between",
        "message0": "Any Between\n char: %0False and_char: %1",
        "previousStatement": "Regular",
        "nextStatement": "Regular",
        "colour": 330,
        "tooltip": "Match any char between `char` and `and_char`, using the ASCII table for reference\n\nArgs:\n    char (str): the first character\n    and_char (str): the second character\n"
    },
    {
        "type": "any_char_except",
        "message0": "Any Char Except\n chars: %0",
        "previousStatement": "Regular",
        "nextStatement": "Regular",
        "colour": 330,
        "tooltip": "This matches any char that is NOT in `chars`. `chars` can be multiple parameters,\nor a single string of chars to split.\n\nArgs:\n    chars (str): any of the characters to match\n"
    },
    {
        "type": "any_of",
        "message0": "Any Of\n patterns: %0",
        "previousStatement": "Regular",
        "nextStatement": "Regular",
        "colour": 330,
        "tooltip": "Ma

In [ ]:
str(ez.chunk(greedy=False, possessive=False))

'.+?'

In [9]:
ez.at_least_one('.', greedy=True, possessive=True).str()

'\\.++'

In [5]:

# inspect.signature(ez.at_least_one._func_list[0]).parameters
for name, param in inspect.signature(ez.at_least_one._func_list[0]).parameters.items():
    # print(name, param)
    if name in ('args', 'kwargs', 'cur'):
        pass
    elif name.endswith('pattern'):
        pass
    elif param.annotation == bool:
        pass
    elif param.annotation == str:
        pass
    elif param.annotation == int:
        pass
    else:
        print(name, param)

In [90]:
ez.int_or_float

PythonEZRegex(..., {'_compiled': None, 'flags': set(), 'replacement': False})

In [38]:
# def get_members(obj):
#     return {i: getattr(obj, i) for i in dir(obj) if i.startswith('__')}

display(dir(ez.any_between._func_list[0]))
re.match(r"(\w+)Mixin\.", ez.any_between._func_list[0].__qualname__).group(1)
# ez.any_between._func_list[0].__doc__
# ez.any_between.#_func_list[0].__doc__


['__annotate__',
 '__annotations__',
 '__builtins__',
 '__call__',
 '__class__',
 '__closure__',
 '__code__',
 '__defaults__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__get__',
 '__getattribute__',
 '__getstate__',
 '__globals__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__kwdefaults__',
 '__le__',
 '__lt__',
 '__module__',
 '__name__',
 '__ne__',
 '__new__',
 '__qualname__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__type_params__']

'Base'

In [32]:
for i in ['__annotate__',
 '__annotations__',
#  '__builtins__',
 '__call__',
 '__class__',
 '__closure__',
 '__code__',
 '__defaults__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__get__',
 '__getattribute__',
 '__getstate__',
 '__globals__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__kwdefaults__',
 '__le__',
 '__lt__',
 '__module__',
 '__name__',
 '__ne__',
 '__new__',
 '__qualname__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__type_params__']:
    print(i, getattr(ez.any_between._func_list[0], i), sep=': ')

__annotate__: <function BaseMixin.<locals>._BaseMixin.__annotate__ at 0x7b3bef0fd010>
__annotations__: {'char': <class 'str'>, 'and_char': <class 'str'>}
__call__: <method-wrapper '__call__' of function object at 0x7b3bef0fd0c0>
__class__: <class 'function'>
__closure__: None
__code__: <code object any_between at 0x7b3bef0f42d0, file "/home/zeke/hello/ezregex/ezregex/mixins.py", line 285>
__defaults__: (Ellipsis,)
__delattr__: <method-wrapper '__delattr__' of function object at 0x7b3bef0fd0c0>
__dict__: {}
__dir__: <built-in method __dir__ of function object at 0x7b3bef0fd0c0>
__doc__: Match any char between `char` and `and_char`, using the ASCII table for reference

Args:
    char (str): the first character
    and_char (str): the second character

__eq__: <method-wrapper '__eq__' of function object at 0x7b3bef0fd0c0>
__format__: <built-in method __format__ of function object at 0x7b3bef0fd0c0>
__ge__: <method-wrapper '__ge__' of function object at 0x7b3bef0fd0c0>
__get__: <method-wra

In [ ]:
cls = ez.PythonEZRegex
include_psuedonyms = False
include_functions = True

parent_members = dir(ez.EZRegex)
[i
    for i in dir(cls)
    if (
        i not in parent_members and
        not i.startswith('__') and
        i not in cls._exclusions and
        i not in cls._deleted and
        (include_psuedonyms or
            (
                i not in ez._all_psuedonyms and
                # If it has an uppercase character, it's a camelCase psuedonym, so remove it
                not any(c.isupper() for c in i)
            )
        ) and
        # (include_psuedonyms or i in ez._psuedonyms) and
        (include_functions or i not in ('options', 'replace'))
        # True
        # (include_compound or i not in ())
        # Because we use this before singleton members are instantiated, getattr().replacement will fail
        # So in that case, we just skip it
        # (
        #     not _check_replacement or
        #     i == 'options' or
        #     include_replacement == (getattr(cls, i).replacement in (True, None))
        # )
    )
]

['any_between',
 'any_char_except',
 'any_of',
 'anything',
 'at_least_none',
 'at_least_one',
 'carriage_return',
 'chunk',
 'comma',
 'controller',
 'digit',
 'each',
 'earlier_group',
 'either',
 'email',
 'form_feed',
 'full_float',
 'group',
 'hex_digit',
 'if_enclosed_with',
 'if_not_preceded_by',
 'if_not_proceded_by',
 'if_preceded_by',
 'if_proceded_by',
 'int_or_float',
 'is_exactly',
 'letter',
 'letter_num',
 'line_ends_with',
 'line_starts_with',
 'literal',
 'literally_anything',
 'lowercase',
 'match_at_least',
 'match_at_most',
 'match_max',
 'match_more_than',
 'match_num',
 'match_range',
 'new_line',
 'not_digit',
 'not_whitespace',
 'not_word',
 'not_word_boundary',
 'number',
 'oct_digit',
 'optional',
 'options',
 'or',
 'ow',
 'passive_group',
 'period',
 'plain_float',
 'printable',
 'printable_and_space',
 'punctuation',
 'quote',
 'raw',
 'replace_entire',
 'rgroup',
 'rliteral',
 'signed_integer',
 'signed_number',
 'space',
 'space_or_tab',
 'string_ends_wit

In [32]:
b

['alphaNum',
 'amtBetween',
 'anyAmt',
 'anyBetween',
 'anyChar',
 'anyCharExcept',
 'anyExcept',
 'anyOf',
 'any_between',
 'any_char_except',
 'any_of',
 'anything',
 'anythingExcept',
 'atLeast',
 'atLeast0',
 'atLeast1',
 'atLeastNone',
 'atLeastOne',
 'atMost',
 'at_least_none',
 'at_least_one',
 'carriageReturn',
 'carriage_return',
 'chunk',
 'comma',
 'controller',
 'digit',
 'each',
 'earlierGroup',
 'earlier_group',
 'either',
 'email',
 'formFeed',
 'form_feed',
 'fullFloat',
 'full_float',
 'group',
 'hexDigit',
 'hex_digit',
 'ifEnclosedBy',
 'ifEnclosedWith',
 'ifExists',
 'ifFollowedBy',
 'ifNotFollowedBy',
 'ifNotPrecededBy',
 'ifNotProcededBy',
 'ifPrecededBy',
 'ifProcededBy',
 'if_enclosed_with',
 'if_not_preceded_by',
 'if_not_proceded_by',
 'if_preceded_by',
 'if_proceded_by',
 'intOrFloat',
 'int_or_float',
 'isExactly',
 'is_exactly',
 'letter',
 'letterNum',
 'letter_num',
 'lineEnd',
 'lineEndsWith',
 'lineStart',
 'lineStartsWith',
 'line_ends_with',
 'line_st

In [30]:
set(a).symmetric_difference(set(b))

{'alpha',
 'alpha_num',
 'alphanum',
 'amt',
 'amt_between',
 'any_amt',
 'any_char',
 'any_except',
 'anychar',
 'anyof',
 'anything_except',
 'at_least',
 'at_least_0',
 'at_least_1',
 'at_most',
 'between',
 'char',
 'dot',
 'exactly',
 'hex',
 'if_enclosed_by',
 'if_followed_by',
 'if_not_followed_by',
 'integer',
 'line_end',
 'line_start',
 'match_amt',
 'match_between',
 'match_greater_than',
 'match_min',
 'more_than',
 'newline',
 'none_or_more',
 'num',
 'num_between',
 'one_of',
 'one_or_more',
 'one_or_none',
 'oneof',
 'opt',
 'or_',
 'repeat',
 'replace_all',
 'replace_group',
 'same_as',
 'same_as_group',
 'signed',
 'signed_int',
 'string_end',
 'string_start',
 'stuff',
 'unsigned',
 'unsigned_int',
 'white_chunk',
 'white_space',
 'whitechar',
 'whitespace',
 'zero_or_more'}

In [ ]:
# Cast function, before I realized SubPattern.dump() exists

# Just freaking cast all iterables (like sre.SubPattern) to lists
def cast(obj):
    try:
        iterable = iter(obj)
    except TypeError:
        if type(obj) == int:
            try:
                # return chr(obj), obj
                return obj
            except ValueError: pass
        return obj
    else:
        return [cast(i) for i in obj]

# pprint(live_log[0][0])
# pprint(cast(live_log[0][0]))

In [ ]:
def castSubPatterns(obj):
    try:
        _ = iter(obj)
    except TypeError:
        if type(obj) == int:
            try:
                # return chr(obj), obj
                return obj
            except ValueError: pass
        return obj
    else:
        return [cast(i) for i in obj]

In [ ]:
# Compile a partial AST

custom = sre.parse(r"\w*")
# ll = live_log[0]

# Mimmics re._compile.compile()
# Compile from an existing AST, instead of a string, so we can test if the partially deconstructed
# AST matches the partial AST it was deconstructed from
# FOR DEBUGGING ONLY
def compile_ast(pattern, flags=re.NOFLAG):
    if not isinstance(pattern, sre.SubPattern):
        return

    p = pattern
    code = re._compiler._code(p, re.NOFLAG)

    # map in either direction -- I don't know what this means or does
    groupindex = p.state.groupdict
    indexgroup = [None] * p.state.groups
    for k, i in groupindex.items():
        indexgroup[i] = k

    return re._compiler._sre.compile(
        # We don't have the actual string, but I think it's just for display, it seems to still work
        "", re.NOFLAG | p.state.flags, code,
        p.state.groups-1,
        groupindex, tuple(indexgroup)
        )

if (partial := compile_ast(custom)):
    partial.search('asdf')

In [ ]:
# Monkey patch _handle
# Don't rerun the imports cell after this, or it will enter an infinite loop!
live_log = []

def new_handle(self, pattern, amt=1, opposite=False):
    logging.debug(f'Handling {pattern} * {amt}')
    live_log.append((pattern, amt, opposite))
    rtn = original_handle(self, pattern, amt, opposite)
    logging.debug(f'Pattern is currently: "{rtn}"')
    # Check that the partial ast we just deconstructed matches the output we just got
    if (partial := compile_ast(pattern, re.NOFLAG)):
        if not (match := partial.search(rtn)):
            logging.warning('Pattern failed to match!')
        else:
            if match.span() != (0, len(rtn)):
                logging.warning('Pattern matched partially!')
            else:
                logging.debug('Valid!')
    return rtn

Inverter._handle = new_handle

In [ ]:
# Invert a partial AST directly to see if it's the issue

# Mimics Inverter.invert_re_parser()
inv = Inverter('')
inv.groups = {}
inv.seed=0.5569052451219174
def sub(l):
    return sre.SubPattern(sre.State(), data=l)
# You can copy these from the log, but be sure to add sre. to the tags, and cast the lists to SubPatterns
# partial_ast = [(sre.IN, [(sre.RANGE, (97, 122)), (sre.RANGE, (48, 57)), (sre.LITERAL, 45)])]
partial_ast = [(sre.MAX_REPEAT, (0, sre.MAXREPEAT, sub([(sre.IN, [(sre.RANGE, (97, 122)), (sre.RANGE, (48, 57)), (sre.LITERAL, 45)])]))), (sre.IN, [(sre.RANGE, (97, 122)), (sre.RANGE, (48, 57))])]

pattern = re._parser.SubPattern(re._parser.State(), data=partial_ast)
# Moneky patched version
for _ in range(1):
    inv._handle(pattern)


In [9]:
# Inverts a pattern directly, with a seed
import ezregex as ez

# pattern = ez.email
# pattern = r"(?:df){3}"
# pattern = r'(?:\w+\s*,\s*)?(\w+),?\s*'
pattern = r"(a??) a*? a{3,}? ab{4,7}?"

# Mimics Inverter.invert_re_parser()
inv = Inverter('')
inv.groups = {}
inv.seed=0.5569052451219174

# Moneky patched version
for _ in range(1):
    inv._handle(sre.parse(str(pattern)))
print(str(pattern))
print(sre.parse(str(pattern)))


DEBUG:root:Handling [(SUBPATTERN, (1, 0, 0, [(MIN_REPEAT, (0, 1, [(LITERAL, 97)]))])), (LITERAL, 32), (MIN_REPEAT, (0, MAXREPEAT, [(LITERAL, 97)])), (LITERAL, 32), (MIN_REPEAT, (3, MAXREPEAT, [(LITERAL, 97)])), (LITERAL, 32), (LITERAL, 97), (MIN_REPEAT, (4, 7, [(LITERAL, 98)]))] * 1
DEBUG:root:Handling [(MIN_REPEAT, (0, 1, [(LITERAL, 97)]))] * 1
DEBUG:root:Handling [(LITERAL, 97)] * 1
DEBUG:root:Literal: a * 1
DEBUG:root:Pattern is currently: "a"
DEBUG:root:Valid!
DEBUG:root:Pattern is currently: "a"
DEBUG:root:group: 1, num: 0, num2: 0 -> s: a -- sub: [(MIN_REPEAT, (0, 1, [(LITERAL, 97)]))]
DEBUG:root:Literal:   * 1
DEBUG:root:Handling [(LITERAL, 97)] * 1
DEBUG:root:Literal: a * 1
DEBUG:root:Pattern is currently: "a"
DEBUG:root:Valid!
DEBUG:root:Literal:   * 1
DEBUG:root:Handling [(LITERAL, 97)] * 1
DEBUG:root:Literal: a * 1
DEBUG:root:Pattern is currently: "a"
DEBUG:root:Valid!
DEBUG:root:Literal:   * 1
DEBUG:root:Literal: a * 1
DEBUG:root:Handling [(LITERAL, 98)] * 1
DEBUG:root:Lite

(a??) a*? a{3,}? ab{4,7}?
[(SUBPATTERN, (1, 0, 0, [(MIN_REPEAT, (0, 1, [(LITERAL, 97)]))])), (LITERAL, 32), (MIN_REPEAT, (0, MAXREPEAT, [(LITERAL, 97)])), (LITERAL, 32), (MIN_REPEAT, (3, MAXREPEAT, [(LITERAL, 97)])), (LITERAL, 32), (LITERAL, 97), (MIN_REPEAT, (4, 7, [(LITERAL, 98)]))]


In [10]:
# Show an AST for a particular regex string
sre.parse(r"(a??) a*? a{3,}? ab{4,7}?").dump()

SUBPATTERN 1 0 0
  MIN_REPEAT 0 1
    LITERAL 97
LITERAL 32
MIN_REPEAT 0 MAXREPEAT
  LITERAL 97
LITERAL 32
MIN_REPEAT 3 MAXREPEAT
  LITERAL 97
LITERAL 32
LITERAL 97
MIN_REPEAT 4 7
  LITERAL 98


In [16]:
# Just figure out what it's supposed to match
re.search(r"(a??) a*? a{3,}? ab{4,7}?", 'a aaaaa aaa abbbbbb')

<re.Match object; span=(0, 17), match='a aaaaa aaa abbbb'>

In [17]:
# Run some tests
live_log.clear()

print(ez.invert(ez.raw(r'(?:df){3}'), backend='re_parser'))

DEBUG:root:re_parser attempt #1 with seed 0.41139557224306644...
DEBUG:root:Literal: d * 1
DEBUG:root:Literal: f * 1
INFO:root:Found using re_parser


dfdfdf


In [20]:
# Disassemble a regex
# Not actually all that useful, but cool, for sure

# Dissassemble a regex
re._compiler.dis(re._compiler._code(sre.parse(r"[a-z0-9!#$%&'*+/=?^_`{|}~-]+"), re.NOFLAG))
# Dissassemble the live log
# re._compiler.dis(re._compiler._code(live_log[0][0], re.NOFLAG))

 0. INFO 4 0b0 1 MAXREPEAT (to 5)
 5: REPEAT_ONE 16 1 MAXREPEAT (to 22)
 9.   IN 11 (to 21)
11.     CHARSET [0x00000000, 0xa3ffacfa, 0xc0000000, 0x7fffffff, 0x00000000, 0x00000000, 0x00000000, 0x00000000]
20.     FAILURE
21:   SUCCESS
22: SUCCESS


In [ ]:
# Also kinda cool, not all that useful
def graphviz_nested(obj):
    dot = Digraph()

    seen_containers = {}
    next_id = 0

    def new_node():
        nonlocal next_id
        nid = f"n{next_id}"
        next_id += 1
        return nid

    def visit(obj):
        # Primitive values: always make a fresh node
        if isinstance(obj, (str, bytes, int, float, bool, type(None))):
            nid = new_node()
            dot.node(nid, repr(obj))
            return nid

        oid = id(obj)

        # Containers/objects: preserve identity
        if oid in seen_containers:
            return seen_containers[oid]

        nid = new_node()
        seen_containers[oid] = nid

        dot.node(nid, type(obj).__name__)

        if isinstance(obj, Mapping):
            for k, v in obj.items():
                k_id = visit(k)
                v_id = visit(v)
                dot.edge(nid, k_id, label="key")
                dot.edge(nid, v_id, label="value")

        elif isinstance(obj, (list, tuple)):
            for i, child in enumerate(obj):
                child_id = visit(child)
                dot.edge(nid, child_id, label=str(i))

        elif isinstance(obj, (set, frozenset)):
            for child in obj:
                child_id = visit(child)
                dot.edge(nid, child_id)

        elif hasattr(obj, "__dict__"):
            for name, value in vars(obj).items():
                child_id = visit(value)
                dot.edge(nid, child_id, label=name)

        return nid

    visit(obj)
    return dot

graphviz_nested(cast(live_log[0][0]))